# Tutorial 03: Two-Part Wasserstein Causal Forests for Structural Zeros

When outcome distributions have a non-trivial point mass at zero ($Q_i = \mathbf{0}_K$), `ZIPTAdapter` (`cwdb_zipt`) combines:
1. Cross-fitted per-arm positive-outcome classifiers $\widehat{\pi}_a(x) = \widehat{\mathbb{P}}(I_i = 1 \mid X_i = x, A_i = a)$, and
2. Positive-stratum CWDB particle boosting $\widehat{P}_{a,+}(x)$,
assembling the two-part mixture law $\widehat{P}_{a,\text{ZI}}^K(x) = (1 - \widehat{\pi}_a(x))\delta_{\mathbf{0}_K} + \widehat{\pi}_a(x)\widehat{P}_{a,+}(x)$ and calibrating functional contrasts via $h(Q_i) = h(\mathbf{0}_K) + I_i(h(Q_i) - h(\mathbf{0}_K))$.

In [ ]:
import os
import sys
from pathlib import Path

if not Path("src/wasserstein_causal_forests").exists() and Path("../src/wasserstein_causal_forests").exists():
    os.chdir("..")
if Path("src").exists() and str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

In [ ]:
import json
from pathlib import Path
import numpy as np

from wasserstein_causal_forests.g3 import runner
from wasserstein_causal_forests.g3.confirmatory_evaluation import evaluate_confirmatory
from wasserstein_causal_forests.g3.manifest import TRAINING_FUNCTIONALS, Cell
from wasserstein_causal_forests.g3.phase65_dgps import register_phase65_dgps
from wasserstein_causal_forests.g3.phase65_methods import ZIPTAdapter

register_phase65_dgps()
cell = Cell("tutorial", "ZI1", 400, 25, 10, "cwdb_zipt", 0)
dgp = runner.build_dgp(cell.dgp, cell.n_grid)
train = dgp.sample(cell.n_train, seed=cell.seed)
test = dgp.sample(300, seed=cell.test_seed)

adapter = ZIPTAdapter(
    n_particles=10,
    classifier_c=1.0,
    n_folds=3,
    contrast_candidates=(0.0, 50.0, 500.0),
    n_estimators=60,
    learning_rate=0.12,
    max_depth=4,
    min_samples_leaf=10,
    min_arm_leaf=5,
    collision_epsilon=1e-3,
)
output = adapter.fit_predict(train=train, X_test=test.X, dgp=dgp, functionals=TRAINING_FUNCTIONALS, seed=cell.seed)
rows = evaluate_confirmatory(cell, output, train, dgp, test.X, runner.evaluation_manifest(cell.n_grid))

by_metric = {}
for row in rows:
    if row["status"] == "ok" and row["value"] is not None:
        by_metric.setdefault(row["metric"], []).append(float(row["value"]))

summary = {k: float(np.mean(v)) for k, v in by_metric.items()}
for key in (
    "kernel_law_error",
    "mean_quantile_rmse",
    "aipw_tate_functional_rmse",
    "aipw_tcate_functional_rmse",
    "aipw_reference_effect_rmse",
    "aipw_reference_tcate_rmse",
):
    print(f"{key:<36}: {summary[key]:.5f}")

output_file = "tutorial_03_zipt_metrics.json"
Path(output_file).write_text(json.dumps(summary, indent=2), encoding="utf-8")

In [ ]:
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)